# Part D: Food-Order Stream with RabbitMQ

## D1. Routing on Paper (6 Marks)

RabbitMQ topic exchange uses two wildcard characters:

- `*` matches exactly one word.
- `#` matches zero or more words.

### Driver Binding Patterns

| Driver | Binding Pattern |
|---|---|
| Ravi | `bangalore.koramangala.*` |
| Priya | `bangalore.indiranagar.*` |
| Arjun | `bangalore.*.ride` |
| Meena | `bangalore.*.food` |
| Karthik | `chennai.tnagar.food` |
| Control | `bangalore.#` |

### Routing Prediction Table

| No. | Routing Key | Receiving Queues |
|---|---|---|
| 1 | `bangalore.koramangala.food` | Ravi, Meena, Control |
| 2 | `bangalore.indiranagar.ride` | Priya, Arjun, Control |
| 3 | `bangalore.whitefield.ride` | Arjun, Control |
| 4 | `chennai.tnagar.food` | Karthik |
| 5 | `chennai.whitefield.ride` | Dropped |
| 6 | `bangalore.koramangala` | Control |

### Explanation of Question 6

The routing key `bangalore.koramangala` contains only two words.

The patterns containing `*` require exactly one word at the wildcard position. The patterns for Ravi, Priya, Arjun and Meena therefore cannot match this two-word key. Karthik's pattern cannot match because it starts with `chennai`.

The Control binding pattern is `bangalore.#`. The `#` wildcard matches zero or more words, so it can match the key even when there are no additional words after `bangalore.koramangala`.

Therefore, only the Control queue receives this message.

**D1 conclusion:** RabbitMQ topic-based routing delivers messages to every queue whose binding pattern matches the routing key. Messages with no matching binding are dropped.

In [1]:
# D2 - Step 1: Install RabbitMQ and Pika
!apt-get update -qq
!apt-get install -y rabbitmq-server
!pip -q install pika

# Start RabbitMQ service
!service rabbitmq-server start

# Verify broker
!rabbitmqctl status

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  erlang-asn1 erlang-base erlang-crypto erlang-eldap erlang-ftp erlang-inets
  erlang-mnesia erlang-os-mon erlang-parsetools erlang-public-key
  erlang-runtime-tools erlang-snmp erlang-ssl erlang-syntax-tools erlang-tftp
  erlang-tools erlang-xmerl libsctp1 logrotate
Suggested packages:
  erlang erlang-manpages erlang-doc lksctp-tools bsd-mailx | mailx
The following NEW packages will be installed:
  erlang-asn1 erlang-base erlang-crypto erlang-eldap erlang-ftp erlang-inets
  erlang-mnesia erlang-os-mon erlang-parsetools erlang-public-key
  erlang-runtime-tools erlang-snmp erlang-ssl erlang-syntax-tools erlang-tftp
  erlang-tools erlang-xmerl libsctp1 lo

In [2]:
import os

os.makedirs("/content/stream", exist_ok=True)

print("Stream folder created.")

Stream folder created.


In [3]:
%%writefile /content/stream/dispatcher.py

import pika
import json
import time
import random
import uuid
from datetime import datetime, timezone

EXCHANGE = "rapido-dispatch"

CITIES = {
    "bangalore": ["koramangala", "indiranagar", "whitefield"],
    "chennai": ["tnagar", "adyar", "velachery"]
}

ORDER_TYPES = ["ride", "food"]

def main():
    connection = pika.BlockingConnection(
        pika.ConnectionParameters(host="localhost")
    )

    channel = connection.channel()

    channel.exchange_declare(
        exchange=EXCHANGE,
        exchange_type="topic",
        durable=True
    )

    print("Dispatcher started. Publishing every 2 seconds...")

    message_count = 0

    try:
        while True:
            city = random.choice(list(CITIES.keys()))
            zone = random.choice(CITIES[city])
            order_type = random.choice(ORDER_TYPES)

            routing_key = f"{city}.{zone}.{order_type}"

            message = {
                "id": str(uuid.uuid4())[:8],
                "city": city,
                "zone": zone,
                "type": order_type,
                "ts": datetime.now(timezone.utc).isoformat()
            }

            channel.basic_publish(
                exchange=EXCHANGE,
                routing_key=routing_key,
                body=json.dumps(message),
                properties=pika.BasicProperties(
                    content_type="application/json",
                    delivery_mode=1
                )
            )

            message_count += 1

            print(
                f"[{message_count}] Published: "
                f"{routing_key} | {message}"
            )

            time.sleep(2)

    except KeyboardInterrupt:
        print("\nDispatcher stopped.")

    finally:
        connection.close()


if __name__ == "__main__":
    main()

Writing /content/stream/dispatcher.py


In [4]:
%%writefile /content/stream/driver.py

import pika
import json
import sys

EXCHANGE = "rapido-dispatch"

def main():
    if len(sys.argv) != 3:
        print("Usage: python driver.py NAME PATTERN")
        sys.exit(1)

    name = sys.argv[1]
    pattern = sys.argv[2]

    connection = pika.BlockingConnection(
        pika.ConnectionParameters(host="localhost")
    )

    channel = connection.channel()

    channel.exchange_declare(
        exchange=EXCHANGE,
        exchange_type="topic",
        durable=True
    )

    result = channel.queue_declare(
        queue="",
        exclusive=True
    )

    queue_name = result.method.queue

    channel.queue_bind(
        exchange=EXCHANGE,
        queue=queue_name,
        routing_key=pattern
    )

    channel.basic_qos(prefetch_count=1)

    print(f"\nDriver: {name}")
    print(f"Binding pattern: {pattern}")
    print(f"Private queue: {queue_name}")
    print("Waiting for requests...\n")

    def callback(ch, method, properties, body):
        try:
            message = json.loads(body.decode("utf-8"))

            print(
                f"[{name}] Received "
                f"{method.routing_key} | {message}"
            )

            # Acknowledge successfully processed message
            ch.basic_ack(delivery_tag=method.delivery_tag)

        except Exception as error:
            print(f"[{name}] Processing error: {error}")
            ch.basic_nack(
                delivery_tag=method.delivery_tag,
                requeue=False
            )

    channel.basic_consume(
        queue=queue_name,
        on_message_callback=callback,
        auto_ack=False
    )

    try:
        channel.start_consuming()

    except KeyboardInterrupt:
        print(f"\nDriver {name} stopped.")

    finally:
        if channel.is_open:
            channel.close()
        connection.close()


if __name__ == "__main__":
    main()

Writing /content/stream/driver.py


In [5]:
import os

for file in [
    "/content/stream/dispatcher.py",
    "/content/stream/driver.py"
]:
    print(file, "Exists:", os.path.exists(file))

/content/stream/dispatcher.py Exists: True
/content/stream/driver.py Exists: True


In [6]:
import subprocess
import time
import os

PYTHON = "/usr/bin/python3"

# Install pika for system Python if needed
subprocess.run(
    [PYTHON, "-m", "pip", "install", "pika", "--break-system-packages", "-q"],
    check=True
)

# Start three drivers
drivers = [
    ("Ravi", "bangalore.koramangala.*"),
    ("Arjun", "bangalore.*.ride"),
    ("Meena", "bangalore.*.food")
]

processes = []
log_handles = []

for name, pattern in drivers:
    log = open(f"/content/{name.lower()}_driver.log", "w")
    log_handles.append(log)

    process = subprocess.Popen(
        [PYTHON, "/content/stream/driver.py", name, pattern],
        stdout=log,
        stderr=subprocess.STDOUT
    )

    processes.append(process)

    print(f"Started driver: {name}")

# Allow drivers to establish their queues
time.sleep(5)

# Start dispatcher
dispatcher_log = open("/content/dispatcher.log", "w")
log_handles.append(dispatcher_log)

dispatcher = subprocess.Popen(
    [PYTHON, "/content/stream/dispatcher.py"],
    stdout=dispatcher_log,
    stderr=subprocess.STDOUT
)

print("Dispatcher started.")
print("Drivers and dispatcher are running.")

Started driver: Ravi
Started driver: Arjun
Started driver: Meena
Dispatcher started.
Drivers and dispatcher are running.


In [7]:
import time

print("Running RabbitMQ stream for 120 seconds...")

for remaining in range(120, 0, -10):
    print(f"{remaining} seconds remaining")
    time.sleep(10)

print("Two-minute streaming period completed.")

Running RabbitMQ stream for 120 seconds...
120 seconds remaining
110 seconds remaining
100 seconds remaining
90 seconds remaining
80 seconds remaining
70 seconds remaining
60 seconds remaining
50 seconds remaining
40 seconds remaining
30 seconds remaining
20 seconds remaining
10 seconds remaining
Two-minute streaming period completed.


In [8]:
for process in [dispatcher] + processes:
    if process.poll() is None:
        process.terminate()

for process in [dispatcher] + processes:
    try:
        process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        process.kill()

for log in log_handles:
    log.close()

print("Dispatcher and all drivers stopped.")

Dispatcher and all drivers stopped.


In [9]:
from pathlib import Path

for filename in [
    "dispatcher.log",
    "ravi_driver.log",
    "arjun_driver.log",
    "meena_driver.log"
]:
    path = Path("/content") / filename

    print("\n" + "=" * 65)
    print(filename)
    print("=" * 65)

    if path.exists():
        content = path.read_text(errors="replace")
        print(content[-5000:])


dispatcher.log
: 'e3a73c73', 'city': 'chennai', 'zone': 'tnagar', 'type': 'ride', 'ts': '2026-11-15T11:32:59.609352+00:00'}
[22] Published: chennai.adyar.ride | {'id': '030f93b0', 'city': 'chennai', 'zone': 'adyar', 'type': 'ride', 'ts': '2026-11-15T11:33:01.609829+00:00'}
[23] Published: chennai.adyar.food | {'id': '6280f98b', 'city': 'chennai', 'zone': 'adyar', 'type': 'food', 'ts': '2026-11-15T11:33:03.610409+00:00'}
[24] Published: bangalore.koramangala.ride | {'id': 'ee799d18', 'city': 'bangalore', 'zone': 'koramangala', 'type': 'ride', 'ts': '2026-11-15T11:33:05.610896+00:00'}
[25] Published: bangalore.whitefield.food | {'id': 'ce76f3cd', 'city': 'bangalore', 'zone': 'whitefield', 'type': 'food', 'ts': '2026-11-15T11:33:07.611459+00:00'}
[26] Published: bangalore.whitefield.ride | {'id': '7e964d97', 'city': 'bangalore', 'zone': 'whitefield', 'type': 'ride', 'ts': '2026-11-15T11:33:09.611995+00:00'}
[27] Published: chennai.tnagar.ride | {'id': 'bf82c217', 'city': 'chennai', 'zone

In [10]:
import subprocess
import time
import sys

PYTHON = "/usr/bin/python3"

# Start drivers with unbuffered output
driver_configs = [
    ("Ravi", "bangalore.koramangala.*"),
    ("Arjun", "bangalore.*.ride"),
    ("Meena", "bangalore.*.food")
]

processes = []
log_handles = []

for name, pattern in driver_configs:
    log = open(f"/content/{name.lower()}_driver.log", "w", buffering=1)
    log_handles.append(log)

    p = subprocess.Popen(
        [
            PYTHON, "-u",
            "/content/stream/driver.py",
            name, pattern
        ],
        stdout=log,
        stderr=subprocess.STDOUT
    )

    processes.append(p)
    print(f"Started {name}: PID {p.pid}")

# Allow queues to be declared and bound
time.sleep(5)

# Start dispatcher
dispatcher_log = open(
    "/content/dispatcher.log", "w", buffering=1
)
log_handles.append(dispatcher_log)

dispatcher = subprocess.Popen(
    [PYTHON, "-u", "/content/stream/dispatcher.py"],
    stdout=dispatcher_log,
    stderr=subprocess.STDOUT
)

print("\nDispatcher started.")
print("Streaming for 120 seconds...")

try:
    for remaining in range(120, 0, -10):
        print(f"{remaining} seconds remaining")
        time.sleep(10)
finally:
    for p in [dispatcher] + processes:
        if p.poll() is None:
            p.terminate()

    for p in [dispatcher] + processes:
        try:
            p.wait(timeout=5)
        except subprocess.TimeoutExpired:
            p.kill()

    for f in log_handles:
        f.close()

print("\nExperiment completed.")

Started Ravi: PID 6431
Started Arjun: PID 6432
Started Meena: PID 6433

Dispatcher started.
Streaming for 120 seconds...
120 seconds remaining
110 seconds remaining
100 seconds remaining
90 seconds remaining
80 seconds remaining
70 seconds remaining
60 seconds remaining
50 seconds remaining
40 seconds remaining
30 seconds remaining
20 seconds remaining
10 seconds remaining

Experiment completed.


In [11]:
from pathlib import Path

for name in ["ravi", "arjun", "meena"]:
    path = Path(f"/content/{name}_driver.log")
    content = path.read_text(errors="replace")

    print("\n" + "=" * 55)
    print(name.upper(), "DRIVER")
    print("=" * 55)
    print(content[:4000] if content else "No messages recorded.")


RAVI DRIVER

Driver: Ravi
Binding pattern: bangalore.koramangala.*
Private queue: amq.gen-EM-NEY2_ZkphOhakvux4Rg
Waiting for requests...

[Ravi] Received bangalore.koramangala.ride | {'id': 'b9af80c8', 'city': 'bangalore', 'zone': 'koramangala', 'type': 'ride', 'ts': '2026-11-15T11:40:33.995419+00:00'}
[Ravi] Received bangalore.koramangala.ride | {'id': 'b525c91e', 'city': 'bangalore', 'zone': 'koramangala', 'type': 'ride', 'ts': '2026-11-15T11:40:45.999566+00:00'}
[Ravi] Received bangalore.koramangala.food | {'id': 'acd4f585', 'city': 'bangalore', 'zone': 'koramangala', 'type': 'food', 'ts': '2026-11-15T11:40:50.000983+00:00'}
[Ravi] Received bangalore.koramangala.food | {'id': '234a4565', 'city': 'bangalore', 'zone': 'koramangala', 'type': 'food', 'ts': '2026-11-15T11:40:54.002707+00:00'}
[Ravi] Received bangalore.koramangala.food | {'id': '883fe99b', 'city': 'bangalore', 'zone': 'koramangala', 'type': 'food', 'ts': '2026-11-15T11:41:04.006019+00:00'}
[Ravi] Received bangalore.koram

In [13]:
from pathlib import Path
from IPython.display import display, HTML

content = Path("/content/dispatcher.log").read_text()

display(HTML(f"""
<div style="
    background-color: #ffffff;
    color: #111111;
    padding: 15px;
    border: 1px solid #cccccc;
    border-radius: 6px;
    font-family: monospace;
    font-size: 13px;
    white-space: pre-wrap;
    overflow-wrap: anywhere;
">
{content[:10000]}
</div>
"""))

In [15]:
%%writefile /content/stream/dispatcher_d3.py
import pika
import json
import time
import random
import uuid
from datetime import datetime, timezone

EXCHANGE = "rapido-dispatch"

CITIES = {
    "bangalore": ["koramangala", "indiranagar", "whitefield"],
    "chennai": ["tnagar", "adyar", "velachery"]
}

ORDER_TYPES = ["ride", "food"]

def main():
    connection = pika.BlockingConnection(
        pika.ConnectionParameters(host="localhost")
    )
    channel = connection.channel()

    channel.exchange_declare(
        exchange=EXCHANGE,
        exchange_type="topic",
        durable=True
    )

    print("D3 Dispatcher started. Publishing every 2 seconds...", flush=True)

    count = 0

    try:
        while True:
            count += 1

            city = random.choice(list(CITIES.keys()))
            zone = random.choice(CITIES[city])
            order_type = random.choice(ORDER_TYPES)

            routing_key = f"{city}.{zone}.{order_type}"

            # Build a normal message first.
            message = {
                "id": str(uuid.uuid4())[:8],
                "city": city,
                "zone": zone,
                "type": order_type,
                "ts": datetime.now(timezone.utc).isoformat()
            }

            # Every 10th message is deliberately malformed:
            # remove the required "zone" field from the message body.
            if count % 10 == 0:
                del message["zone"]
                status = "MALFORMED - zone missing"
            else:
                status = "VALID"

            channel.basic_publish(
                exchange=EXCHANGE,
                routing_key=routing_key,
                body=json.dumps(message),
                properties=pika.BasicProperties(
                    content_type="application/json",
                    delivery_mode=1
                )
            )

            print(
                f"[{count}] {status} | Routing key: {routing_key} | "
                f"Message: {message}",
                flush=True
            )

            time.sleep(2)

    except KeyboardInterrupt:
        print("\nD3 Dispatcher stopped.", flush=True)

    finally:
        connection.close()


if __name__ == "__main__":
    main()

Writing /content/stream/dispatcher_d3.py


In [16]:
%%writefile /content/stream/driver_d3.py
import pika
import json
import sys

EXCHANGE = "rapido-dispatch"

def main():
    if len(sys.argv) != 3:
        print("Usage: python driver_d3.py NAME PATTERN")
        sys.exit(1)

    name = sys.argv[1]
    pattern = sys.argv[2]

    connection = pika.BlockingConnection(
        pika.ConnectionParameters(host="localhost")
    )
    channel = connection.channel()

    channel.exchange_declare(
        exchange=EXCHANGE,
        exchange_type="topic",
        durable=True
    )

    # Each driver receives messages through its own private queue.
    result = channel.queue_declare(queue="", exclusive=True)
    queue_name = result.method.queue

    channel.queue_bind(
        exchange=EXCHANGE,
        queue=queue_name,
        routing_key=pattern
    )

    channel.basic_qos(prefetch_count=1)

    print(f"\nD3 Driver: {name}", flush=True)
    print(f"Binding pattern: {pattern}", flush=True)
    print(f"Private queue: {queue_name}", flush=True)
    print("Waiting for messages...\n", flush=True)

    def callback(ch, method, properties, body):
        try:
            message = json.loads(body.decode("utf-8"))

            # Assignment requirement: reject messages missing "zone".
            if "zone" not in message:
                print(
                    f"[{name}] REJECTED | {method.routing_key} | "
                    f"Missing required field: zone | NACK requeue=False",
                    flush=True
                )
                ch.basic_nack(
                    delivery_tag=method.delivery_tag,
                    requeue=False
                )
                return

            print(
                f"[{name}] PROCESSED | {method.routing_key} | "
                f"Message: {message} | ACK",
                flush=True
            )

            ch.basic_ack(delivery_tag=method.delivery_tag)

        except Exception as error:
            print(
                f"[{name}] REJECTED | Processing error: {error} | "
                f"NACK requeue=False",
                flush=True
            )
            ch.basic_nack(
                delivery_tag=method.delivery_tag,
                requeue=False
            )

    channel.basic_consume(
        queue=queue_name,
        on_message_callback=callback,
        auto_ack=False
    )

    try:
        channel.start_consuming()

    except KeyboardInterrupt:
        print(f"\nDriver {name} stopped.", flush=True)

    finally:
        if channel.is_open:
            channel.close()
        connection.close()


if __name__ == "__main__":
    main()

Writing /content/stream/driver_d3.py


In [17]:
!ls -l /content/stream/dispatcher_d3.py /content/stream/driver_d3.py

-rw-r--r-- 1 root root 2185 Nov 15 11:53 /content/stream/dispatcher_d3.py
-rw-r--r-- 1 root root 2549 Nov 15 11:53 /content/stream/driver_d3.py


In [18]:
!service rabbitmq-server status

Status of node rabbit@2217eefb416c ...
Runtime

OS PID: 2514
OS: Linux
Uptime (seconds): 1627
Is under maintenance?: false
RabbitMQ version: 3.12.1
RabbitMQ release series support status: supported
Node name: rabbit@2217eefb416c
Erlang configuration: Erlang/OTP 25 [erts-13.2.2.5] [source] [64-bit] [smp:2:2] [ds:2:2:10] [async-threads:1] [jit:ns]
Crypto library: OpenSSL 3.0.13 30 Jan 2024
Erlang processes: 273 used, 1048576 limit
Scheduler run queue: 1
Cluster heartbeat timeout (net_ticktime): 60

Plugins

Enabled plugin file: /etc/rabbitmq/enabled_plugins
Enabled plugins:


Data directory

Node data directory: /var/lib/rabbitmq/mnesia/rabbit@2217eefb416c
Raft data directory: /var/lib/rabbitmq/mnesia/rabbit@2217eefb416c/quorum/rabbit@2217eefb416c

Config files


Log file(s)

 * /var/log/rabbitmq/rabbit@2217eefb416c.log
 * <stdout>

Alarms

(none)

Memory

Total memory used: 0.1299 gb
Calculation strategy: rss
Memory high watermark setting: 0.4 of available memory, computed to: 5.4423 gb

In [19]:
!service rabbitmq-server start

 * Starting RabbitMQ Messaging Server rabbitmq-server         * RabbitMQ Messaging Server already running
[ OK ]


In [20]:
import subprocess
import time

base = "/content/stream"

# Start three drivers with the same patterns used in D2.
ravi_log = open("/content/d3_ravi.log", "w")
arjun_log = open("/content/d3_arjun.log", "w")
meena_log = open("/content/d3_meena.log", "w")
dispatcher_log = open("/content/d3_dispatcher.log", "w")

ravi = subprocess.Popen(
    ["python", "-u", f"{base}/driver_d3.py",
     "Ravi", "bangalore.koramangala.*"],
    stdout=ravi_log, stderr=subprocess.STDOUT
)

arjun = subprocess.Popen(
    ["python", "-u", f"{base}/driver_d3.py",
     "Arjun", "bangalore.*.ride"],
    stdout=arjun_log, stderr=subprocess.STDOUT
)

meena = subprocess.Popen(
    ["python", "-u", f"{base}/driver_d3.py",
     "Meena", "bangalore.*.food"],
    stdout=meena_log, stderr=subprocess.STDOUT
)

time.sleep(5)  # Allow drivers to connect and bind their queues.

dispatcher = subprocess.Popen(
    ["python", "-u", f"{base}/dispatcher_d3.py"],
    stdout=dispatcher_log, stderr=subprocess.STDOUT
)

print("D3 experiment running for 120 seconds...")
time.sleep(120)

# Stop the dispatcher and drivers cleanly.
dispatcher.terminate()
ravi.terminate()
arjun.terminate()
meena.terminate()

for process in [dispatcher, ravi, arjun, meena]:
    try:
        process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        process.kill()

for log in [dispatcher_log, ravi_log, arjun_log, meena_log]:
    log.close()

print("D3 experiment finished. Logs saved.")

D3 experiment running for 120 seconds...
D3 experiment finished. Logs saved.


In [21]:
from pathlib import Path

files = {
    "Dispatcher": "/content/d3_dispatcher.log",
    "Ravi": "/content/d3_ravi.log",
    "Arjun": "/content/d3_arjun.log",
    "Meena": "/content/d3_meena.log"
}

for name, filepath in files.items():
    content = Path(filepath).read_text()
    print(f"\n{'='*20} {name} {'='*20}")
    print("Total lines:", len(content.splitlines()))
    print("Malformed messages:", content.count("MALFORMED"))
    print("Rejected messages:", content.count("REJECTED"))
    print("Processed messages:", content.count("PROCESSED"))
    print("\nLast output lines:")
    print("\n".join(content.splitlines()[-8:]))


==================== Dispatcher ====================
Total lines: 61
Malformed messages: 6
Rejected messages: 0
Processed messages: 0

Last output lines:
[53] VALID | Routing key: chennai.velachery.food | Message: {'id': '90eb18fb', 'city': 'chennai', 'zone': 'velachery', 'type': 'food', 'ts': '2026-11-15T11:56:26.287514+00:00'}
[54] VALID | Routing key: chennai.adyar.food | Message: {'id': 'c83e3ac2', 'city': 'chennai', 'zone': 'adyar', 'type': 'food', 'ts': '2026-11-15T11:56:28.288065+00:00'}
[55] VALID | Routing key: chennai.tnagar.food | Message: {'id': 'd934bea5', 'city': 'chennai', 'zone': 'tnagar', 'type': 'food', 'ts': '2026-11-15T11:56:30.288601+00:00'}
[56] VALID | Routing key: chennai.adyar.food | Message: {'id': 'eadc3ddc', 'city': 'chennai', 'zone': 'adyar', 'type': 'food', 'ts': '2026-11-15T11:56:32.289325+00:00'}
[57] VALID | Routing key: bangalore.indiranagar.food | Message: {'id': '73965679', 'city': 'bangalore', 'zone': 'indiranagar', 'type': 'food', 'ts': '2026-11-1

Q1. What happens if the driver crashes before sending an ACK?

Because the driver uses manual acknowledgement (auto_ack=False), RabbitMQ considers the message unacknowledged. If the consumer connection closes before ACK, RabbitMQ requeues the unacknowledged message so that another consumer can process it.

Q2. What happens if requeue=True is used for a message that is always bad?

RabbitMQ puts the invalid message back into the queue. The consumer may receive and reject it repeatedly, creating an endless redelivery loop. Using requeue=False discards the message (unless a dead-letter exchange is configured).